INSTALLING AND IMPORTING LIBRARIES

In [1]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

IMPORTING create_react_agent AND AgentExecutor FROM langchain.agents

In [2]:
from langchain.agents import create_react_agent, AgentExecutor

# create_react_agent:
# Creates a ReAct (Reasoning + Acting) Agent that can think, choose tools,
# execute actions, observe results, and generate a final answer.

# AgentExecutor:
# Runs the agent, manages the execution loop, calls tools,
# processes observations, and returns the final response.

LOAD THE ENV VARIABLES

In [4]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
EXCHANGE_RATE_API_KEY = os.getenv("EXCHANGE_RATE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

TAVILY SERACH RESULTS CHECKING

In [5]:
search_tool = TavilySearchResults(max_results=4)

In [6]:
result = search_tool.invoke("Best tourist places in Japan")
result

[{'url': 'https://www.timetravelturtle.com/japan/places-to-visit-in-japan',
  'content': 'Honshu\n  + Tokyo\n  + Kamakura\n  + Mount Fuji\n  + Kyoto\n  + Nara\n  + Osaka\n  + Kanazawa\n  + Aizu-Wakamatsu\n  + Hiroshima\n\n Hokkaido\n  + Sapporo\n  + Hakodate\n  + Ashikawa\n Shikoku\n  + Kochi\n  + Ehime\n  + Art Islands\n Kyushu\n  + Fukuoka\n  + Nagasaki\n  + Oita\n  + Okinawa\n\nIf I could give one bit of advice for visiting Japan, it’s to not consider this to be your only visit. There are so many places to visit in Japan, it’s a country that you can come back to time and time again and always find something new.\n\nFrom the neon cities and vibrant towns to the natural escapes and authentic experiences, Japan’s 47 prefectures each have their own unique offerings in their food, heritage, and culture. [...] ### Ehime\n\nIn the neighbouring prefecture of Ehime, the local mascot is made to look like a mikan, a seedless mandarin that is grown across the whole region. Fertile fields for ag

EXCHANGE RATE CHECKING

In [32]:
@tool
def get_exchange_rate(currency_pair: str) -> str:
    """
    Get exchange rate between two currencies.

    Example:
    INR,JPY
    USD,INR
    EUR,JPY
    """

    from_currency, to_currency = [
        c.strip().upper()
        for c in currency_pair.split(",")
    ]

    url = (
        f"https://v6.exchangerate-api.com/v6/"
        f"{EXCHANGE_RATE_API_KEY}/latest/{from_currency}"
    )

    response = requests.get(url)

    data = response.json()

    rate = data["conversion_rates"][to_currency]

    return f"1 {from_currency} = {rate} {to_currency}"

In [33]:
get_exchange_rate.invoke("INR,JPY")

'1 INR = 1.68 JPY'

In [44]:
print(type(get_exchange_rate))

<class 'langchain_core.tools.StructuredTool'>


CURRENT WEATHER INFORMATION

In [13]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%\n"
        f"Wind Speed: {data['current']['wind_speed']} km/h\n"
        f"Feels Like: {data['current']['feelslike']}°C"
    )

In [15]:
result = get_weather_data.invoke(
    {
        "city": "Bangalore"
    }
)
print(result)

City: Bangalore
Temperature: 30°C
Weather: Partly Cloudy 
Humidity: 46%
Wind Speed: 9 km/h
Feels Like: 30°C


In [16]:
print(type(get_weather_data))

<class 'langchain_core.tools.StructuredTool'>


COUNTRIES INFORMATION

In [17]:
@tool
def get_country_info(country: str) -> str:
    """
    Fetch country information including capital, currency,
    population, region, languages, flag and country codes.
    """

    url = (
        f"https://restcountries.com/v3.1/name/{country}"
        "?fields=name,capital,currencies,population,region,languages,flags,cca2,cca3"
    )

    response = requests.get(url)

    data = response.json()

    if not data or isinstance(data, dict):
        return f"Could not fetch country information for {country}"

    country_data = data[0]

    currency_code = list(country_data["currencies"].keys())[0]
    currency_name = country_data["currencies"][currency_code]["name"]

    languages = ", ".join(country_data["languages"].values())

    return (
        f"Country: {country_data['name']['common']}\n"
        f"Capital: {country_data['capital'][0]}\n"
        f"Region: {country_data['region']}\n"
        f"Population: {country_data['population']:,}\n"
        f"Currency: {currency_name} ({currency_code})\n"
        f"Languages: {languages}\n"
        f"Country Code: {country_data['cca2']} / {country_data['cca3']}\n"
        f"Flag: {country_data['flags']['png']}"
    )

In [19]:
result = get_country_info.invoke("Japan")
print(result)

Country: Japan
Capital: Tokyo
Region: Asia
Population: 123,210,000
Currency: Japanese yen (JPY)
Languages: Japanese
Country Code: JP / JPN
Flag: https://flagcdn.com/w320/jp.png


In [20]:
print(type(get_country_info))

<class 'langchain_core.tools.StructuredTool'>


In [34]:
search_tool = TavilySearchResults(max_results=4)

tools = [
    search_tool,
    get_weather_data,
    get_exchange_rate,
    get_country_info
]

In [35]:
print(tools)

[TavilySearchResults(max_results=4), StructuredTool(name='get_weather_data', description='get_weather_data(city: str) -> str - Fetch current weather information for a city.', args_schema=<class 'pydantic.v1.main.get_weather_dataSchema'>, func=<function get_weather_data at 0x000001388C30DE40>), StructuredTool(name='get_exchange_rate', description='get_exchange_rate(currency_pair: str) -> str - Get exchange rate between two currencies.\n\nExample:\nINR,JPY\nUSD,INR\nEUR,JPY', args_schema=<class 'pydantic.v1.main.get_exchange_rateSchema'>, func=<function get_exchange_rate at 0x0000013886B9A660>), StructuredTool(name='get_country_info', description='get_country_info(country: str) -> str - Fetch country information including capital, currency,\npopulation, region, languages, flag and country codes.', args_schema=<class 'pydantic.v1.main.get_country_infoSchema'>, func=<function get_country_info at 0x00000138837A96C0>)]


In [23]:
print(tools)

for t in tools:
    print(type(t))
    print(t.name)

[TavilySearchResults(max_results=4), StructuredTool(name='get_weather_data', description='get_weather_data(city: str) -> str - Fetch current weather information for a city.', args_schema=<class 'pydantic.v1.main.get_weather_dataSchema'>, func=<function get_weather_data at 0x000001388C30DE40>), StructuredTool(name='get_exchange_rate', description='get_exchange_rate(from_currency: str, to_currency: str) -> str - Get the current exchange rate between two currencies.\n\nExample:\nfrom_currency = USD\nto_currency = INR', args_schema=<class 'pydantic.v1.main.get_exchange_rateSchema'>, func=<function get_exchange_rate at 0x000001388C30EF20>), StructuredTool(name='get_country_info', description='get_country_info(country: str) -> str - Fetch country information including capital, currency,\npopulation, region, languages, flag and country codes.', args_schema=<class 'pydantic.v1.main.get_country_infoSchema'>, func=<function get_country_info at 0x00000138837A96C0>)]
<class 'langchain_community.to

LLM 

In [36]:
# ==========================================
# LLM
# ==========================================

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    max_tokens=500
)

In [25]:
response = llm.invoke("who won 2015 cricket world cup?")
print(response)

content='Australia won the 2015 Cricket World Cup. They defeated New Zealand in the final, which was held at the Melbourne Cricket Ground on March 29, 2015. Australia won the match by seven wickets, claiming their fifth World Cup title.' response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 16, 'total_tokens': 66, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 3.24e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 3.24e-05, 'upstream_inference_prompt_cost': 2.4e-06, 'upstream_inference_completions_cost': 3e-05}}, 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_b7b4c219e8', 'finish_reason': 'stop', 'logprobs': None} id='run-d77b0e41-1e9c-46b7-a392-ae73b65a9d45-0'


REACT PROMPT

In [37]:
# ==========================================
# PROMPT
# ==========================================

prompt = hub.pull("hwchase17/react")           #why I use this prompt? because it is a prompt that is designed to work with the ReAct framework, which allows the agent to reason about its actions and make decisions based on the information it has. This prompt provides a structure for the agent to follow, which can help it to generate more coherent and relevant responses.

c:\Users\Gouthum\Downloads\Complete Agentic AI Coursework\5.1 Agentic-AI-Travel-Planner-Agent\venv\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [38]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

TOOLS

In [39]:
# ==========================================
# TOOLS
# ==========================================

tools = [
    search_tool,
    get_weather_data,
    get_exchange_rate,
    get_country_info
]
print(tools)

[TavilySearchResults(max_results=4), StructuredTool(name='get_weather_data', description='get_weather_data(city: str) -> str - Fetch current weather information for a city.', args_schema=<class 'pydantic.v1.main.get_weather_dataSchema'>, func=<function get_weather_data at 0x000001388C30DE40>), StructuredTool(name='get_exchange_rate', description='get_exchange_rate(currency_pair: str) -> str - Get exchange rate between two currencies.\n\nExample:\nINR,JPY\nUSD,INR\nEUR,JPY', args_schema=<class 'pydantic.v1.main.get_exchange_rateSchema'>, func=<function get_exchange_rate at 0x0000013886B9A660>), StructuredTool(name='get_country_info', description='get_country_info(country: str) -> str - Fetch country information including capital, currency,\npopulation, region, languages, flag and country codes.', args_schema=<class 'pydantic.v1.main.get_country_infoSchema'>, func=<function get_country_info at 0x00000138837A96C0>)]


CREATING THE AGENT

In [40]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

AGENT EXECUTOR

In [41]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(                      #AgentExecutor is a class that takes an agent and a list of tools, and allows you to execute the agent with the tools. It handles the interaction between the agent and the tools, allowing the agent to call the tools as needed to complete its tasks.
    agent=agent,                                     #agent=agent means that we are passing the agent we just created to the AgentExecutor, so that it can use that agent to execute tasks. we use only single agent in this course, so we can just pass that agent to the AgentExecutor.
    tools=tools,                                     #tools=tools means that we are passing the list of tools we created to the AgentExecutor, so that it can use those tools to execute tasks. the agent will be able to call these tools as needed to complete its tasks.
    verbose=True                                     #verbose=True means that we want the AgentExecutor to print out the steps it is taking as it executes tasks. This can be helpful for debugging and understanding how the agent is using the tools to complete its tasks.
)

In [ ]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True                    #handle_parsing_errors=True means that if the agent generates a response that cannot be parsed correctly (for example, if it calls a tool with incorrect arguments), the AgentExecutor will catch that error and allow the agent to try again, rather than crashing the entire execution. This can help to make the agent more robust and able to recover from mistakes in its reasoning or tool usage.
)

RESULT

In [ ]:
# ==========================================
# Result
# ==========================================

response = agent_executor.invoke({
    "input": (
        "I am travelling from India to Japan. "
        "Tell me about Japan including its capital, currency, population and languages. "
        "Convert 10000 INR to Japanese Yen. "
        "Tell me the current weather in Tokyo. "
        "Also suggest the top 5 tourist attractions in Tokyo."
    )
})

print(response["output"])



> Entering new AgentExecutor chain...
I need to gather information about Japan, including its capital, currency, population, and languages. Then, I will convert 10,000 INR to Japanese Yen and check the current weather in Tokyo. Finally, I will look for the top 5 tourist attractions in Tokyo.

Action: get_country_info  
Action Input: Japan  Country: Japan
Capital: Tokyo
Region: Asia
Population: 123,210,000
Currency: Japanese yen (JPY)
Languages: Japanese
Country Code: JP / JPN
Flag: https://flagcdn.com/w320/jp.pngI have gathered information about Japan, including its capital, currency, population, and languages. Now, I will convert 10,000 INR to Japanese Yen.

Action: get_exchange_rate  
Action Input: INR,JPY  1 INR = 1.68 JPYTo convert 10,000 INR to Japanese Yen, I will multiply the amount in INR by the exchange rate.

Calculation:  
10,000 INR * 1.68 JPY/INR = 16,800 JPY

Now, I will check the current weather in Tokyo.

Action: get_weather_data  
Action Input: Tokyo  City: Tokyo
Tem